# Network Congestion Forecasting
This module trains a separate Random Forest model to predict **future** network congestion using temporal features like rolling averages, lags, and trends.

### Why is forecasting separate from classification?
1. **Objectives**: Classification determines the *current* state of the network. Forecasting predicts the *future* state based on historical sequences.
2. **Computational Constraints**: The classification model needs to be ultra-fast and lightweight for real-time ESP8266 simulated data. Forecasting requires calculating rolling windows and maintaining historical state, which is better handled separately.
3. **Data Requirements**: Mixing instantaneous and temporal features can dilute the immediate real-time signals.

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

### 1. Load Data
We load the dataset and ensure it is sorted chronologically, which is essential for any time-series and rolling operations.

In [2]:
# Load dataset
df = pd.read_csv('network_congestion.csv')

# Ensure temporal sorting if timestamp is available
if 'timestamp' in df.columns:
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values(by='timestamp').reset_index(drop=True)

df.head()

,timestamp,hour,active_users,latency,packet_loss,throughput,signal_strength,congestion_level
0,2026-01-01 00:00:00,0,81.897655,79.746673,0.761155,89.426578,-50.378484,0
1,2026-01-01 00:00:05,0,65.740078,64.404862,2.053511,65.041440,-51.862919,0
2,2026-01-01 00:00:10,0,33.926996,48.218296,1.361334,69.259056,-49.430142,0
3,2026-01-01 00:00:15,0,40.202375,54.302033,2.227257,75.088592,-52.542239,0
4,2026-01-01 00:00:20,0,47.243945,54.147506,0.356083,70.732656,-51.786966,0


### 2. Feature Engineering
We will generate rolling means, standard deviations, lags, and trend differences.

In [3]:
window_size = 5  # Rolling window size

# 1. Rolling Features (Averages)
df['users_roll'] = df['active_users'].rolling(window=window_size).mean()
df['latency_roll'] = df['latency'].rolling(window=window_size).mean()
df['throughput_roll'] = df['throughput'].rolling(window=window_size).mean()

# 2. Lag Features (Previous value)
df['users_lag1'] = df['active_users'].shift(1)
df['latency_lag1'] = df['latency'].shift(1)
df['throughput_lag1'] = df['throughput'].shift(1)

# 3. Trend Features (Change from previous step)
df['latency_change'] = df['latency'] - df['latency_lag1']
df['throughput_change'] = df['throughput'] - df['throughput_lag1']

# 4. Rolling Standard Deviation (Volatility)
df['latency_std'] = df['latency'].rolling(window=window_size).std()
df['throughput_std'] = df['throughput'].rolling(window=window_size).std()

df.head(10)

,timestamp,hour,active_users,latency,packet_loss,throughput,signal_strength,congestion_level,users_roll,latency_roll,throughput_roll,users_lag1,latency_lag1,throughput_lag1,latency_change,throughput_change,latency_std,throughput_std
0,2026-01-01 00:00:00,0,81.897655,79.746673,0.761155,89.426578,-50.378484,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-01-01 00:00:05,0,65.740078,64.404862,2.053511,65.041440,-51.862919,0,NaN,NaN,NaN,81.897655,79.746673,89.426578,-15.341810,-24.385137,NaN,NaN
2,2026-01-01 00:00:10,0,33.926996,48.218296,1.361334,69.259056,-49.430142,0,NaN,NaN,NaN,65.740078,64.404862,65.041440,-16.186566,4.217616,NaN,NaN
3,2026-01-01 00:00:15,0,40.202375,54.302033,2.227257,75.088592,-52.542239,0,NaN,NaN,NaN,33.926996,48.218296,69.259056,6.083736,5.829536,NaN,NaN
4,2026-01-01 00:00:20,0,47.243945,54.147506,0.356083,70.732656,-51.786966,0,53.802210,60.163874,73.909664,40.202375,54.302033,75.088592,-0.154526,-4.355936,12.396841,9.387910
5,2026-01-01 00:00:25,0,55.213178,60.722500,1.493734,72.899531,-51.204415,0,48.465314,56.359040,70.604255,47.243945,54.147506,70.732656,6.574994,2.166875,6.308108,3.812759
6,2026-01-01 00:00:30,0,73.156688,171.518644,1.633675,30.882655,-53.373680,0,49.948636,77.781796,63.772498,55.213178,60.722500,72.899531,110.796144,-42.016876,52.586823,18.517860
7,2026-01-01 00:00:35,0,55.076427,63.629079,1.914524,80.779231,-49.305592,0,54.178523,80.863953,66.076533,73.156688,171.518644,30.882655,-107.889565,49.896576,50.843623,20.026410
8,2026-01-01 00:00:40,0,55.966133,68.320314,1.298547,68.550621,-45.281135,0,57.331274,83.667609,64.768939,55.076427,63.629079,80.779231,4.691234,-12.228610,49.378135,19.497328
9,2026-01-01 00:00:45,0,43.994214,65.150868,1.430611,76.419506,-47.055511,0,56.681328,85.868281,65.906309,55.966133,68.320314,68.550621,-3.169445,7.868885,47.958366,20.089083


### 3. Future Target Definition
We shift the `congestion_level` backwards to predict the NEXT time step.

In [4]:
# Shift -1 means the target is the value of congestion_level in the next row
df['future_congestion'] = df['congestion_level'].shift(-1)

# Drop NaNs caused by rolling/shifting
df_clean = df.dropna().reset_index(drop=True)

print(f"Original rows: {len(df)}, Cleaned rows: {len(df_clean)}")

Original rows: 20000, Cleaned rows: 19995


### 4. Train Forecasting Model
We train a new `RandomForestClassifier` purely on these features.

In [5]:
# Define the complete feature set required at runtime
forecasting_features = [
    'active_users', 'latency', 'throughput', 'packet_loss', 'signal_strength',
    'users_roll', 'latency_roll', 'throughput_roll',
    'users_lag1', 'latency_lag1', 'throughput_lag1',
    'latency_change', 'throughput_change',
    'latency_std', 'throughput_std'
]

X = df_clean[forecasting_features]
y = df_clean['future_congestion']

# Time-series split (shuffle=False keeps chronological order)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8932

Classification Report:
              precision    recall  f1-score   support

         0.0       0.95      0.93      0.94      2431
         1.0       0.76      0.85      0.81      1035
         2.0       0.93      0.82      0.87       533

    accuracy                           0.89      3999
   macro avg       0.88      0.87      0.87      3999
weighted avg       0.90      0.89      0.89      3999



### 5. Save Model and Features
We save the model and the list of features so `app.py` knows exactly what to pass in.

In [6]:
joblib.dump(rf_model, 'forecasting_model.pkl')
joblib.dump(forecasting_features, 'forecasting_features.pkl')

print("Saved forecasting_model.pkl and forecasting_features.pkl successfully!")

Saved forecasting_model.pkl and forecasting_features.pkl successfully!
